# MF-Bias Retrain (2x T4) — Recover corrupted state.pt

In [ ]:

!pip install -q pyarrow pandas
import os, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
print('torch', torch.__version__, 'cuda', torch.cuda.device_count())


In [ ]:

def find_parquets(sub_dir):
    patterns = [f"/kaggle/input/**/amazon_recs_output/{sub_dir}/*.parquet",
                f"/kaggle/input/datasets/nguynteo/lastone/amazon_recs_output/{sub_dir}/*.parquet"]
    for p in patterns:
        files = sorted(glob.glob(p, recursive=True))
        if files: return files
    return []
train_files = find_parquets("train")
test_files = find_parquets("test")
print(f"train: {len(train_files)}, test: {len(test_files)}")
dfs = [pq.read_table(f).to_pandas() for f in train_files]
train_df = pd.concat(dfs, ignore_index=True)
print(f"train interactions: {len(train_df):,}")


In [ ]:

users_raw = train_df['user_idx'].to_numpy()
items_raw = train_df['item_idx'].to_numpy()
ratings = train_df['rating'].fillna(0).to_numpy(dtype=np.float32)
u_uniq, u_inv = np.unique(users_raw, return_inverse=True)
i_uniq, i_inv = np.unique(items_raw, return_inverse=True)
U = len(u_uniq); I = len(i_uniq)
global_mean = float(ratings.mean())
print(f"U={U:,} I={I:,} mean={global_mean:.4f}")
u_train_t = torch.from_numpy(u_inv.astype(np.int64))
i_train_t = torch.from_numpy(i_inv.astype(np.int64))
r_train_t = torch.from_numpy(ratings)
train_dataset = TensorDataset(u_train_t, i_train_t, r_train_t)
BATCH = 8192
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)


In [ ]:

FACTORS = 64
EPOCHS = 5
LR = 0.005
WD = 1e-5

class MFBias(nn.Module):
    def __init__(self, n_users, n_items, f, mean_val=0.0):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, f)
        self.item_emb = nn.Embedding(n_items, f)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.tensor(mean_val, dtype=torch.float32))
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
    def forward(self, u, i):
        dot = (self.user_emb(u) * self.item_emb(i)).sum(dim=1)
        return dot + self.user_bias(u).squeeze(-1) + self.item_bias(i).squeeze(-1) + self.global_bias

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MFBias(U, I, FACTORS, mean_val=global_mean)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
loss_fn = nn.MSELoss()
print(f"Model on {device}, {torch.cuda.device_count()} GPUs")


In [ ]:

t_all = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0; total_samples = 0
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, (ub, ib, rb) in enumerate(pbar):
        ub = ub.to(device, non_blocking=True)
        ib = ib.to(device, non_blocking=True)
        rb = rb.to(device, non_blocking=True)
        optimizer.zero_grad()
        pred = model(ub, ib)
        loss = loss_fn(pred, rb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(rb)
        total_samples += len(rb)
        if (step + 1) % 200 == 0:
            rmse = (total_loss / total_samples) ** 0.5
            pbar.set_postfix({"rmse": f"{rmse:.4f}"})
    epoch_rmse = (total_loss / total_samples) ** 0.5
    print(f"[Epoch {epoch+1}/{EPOCHS}] Train RMSE: {epoch_rmse:.4f} | Time: {time.time()-t0:.1f}s", flush=True)
print(f"Training done in {time.time()-t_all:.1f}s")


In [ ]:

# Save model state
m = model.module if hasattr(model, 'module') else model
torch.save({
    'user_emb.weight': m.user_emb.weight.detach().cpu(),
    'item_emb.weight': m.item_emb.weight.detach().cpu(),
    'user_bias.weight': m.user_bias.weight.detach().cpu(),
    'item_bias.weight': m.item_bias.weight.detach().cpu(),
    'global_bias': m.global_bias.detach().cpu(),
}, '/kaggle/working/mf_bias_state.pt')
np.save('/kaggle/working/mf_user_ids.npy', u_uniq)
np.save('/kaggle/working/mf_item_ids.npy', i_uniq)
print("Saved mf_bias_state.pt + mappings")

# Eval on test
dfs_test = [pq.read_table(f).to_pandas() for f in test_files]
test_df = pd.concat(dfs_test, ignore_index=True)
u_map = {uid: idx for idx, uid in enumerate(u_uniq)}
i_map = {iid: idx for idx, iid in enumerate(i_uniq)}
mask = test_df['user_idx'].isin(u_map) & test_df['item_idx'].isin(i_map)
test_valid = test_df[mask]
u_test = np.array([u_map[x] for x in test_valid['user_idx'].values], dtype=np.int64)
i_test = np.array([i_map[x] for x in test_valid['item_idx'].values], dtype=np.int64)
r_true = test_valid['rating'].fillna(0).to_numpy(dtype=np.float32)

model.eval()
preds = []
with torch.no_grad():
    for s in range(0, len(u_test), 500000):
        e = min(s+500000, len(u_test))
        ub = torch.from_numpy(u_test[s:e]).to(device)
        ib = torch.from_numpy(i_test[s:e]).to(device)
        p = model(ub, ib).cpu().numpy()
        preds.append(p)
preds = np.concatenate(preds)
rmse = float(np.sqrt(np.mean((preds - r_true)**2)))
mae = float(np.mean(np.abs(preds - r_true)))
print(f"Test RMSE: {rmse:.4f} | Test MAE: {mae:.4f} | n={len(r_true)}")

metrics = {'rmse': rmse, 'mae': mae, 'factors': FACTORS, 'epochs': EPOCHS, 'n': len(r_true)}
open('/kaggle/working/mf_bias_metrics.json','w').write(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
